<a href="https://colab.research.google.com/github/mdhanush0009-ux/mdhanush2026B03110248/blob/main/travel_booking_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Project 29: Travel Booking & Tourist Preference Analysis
Run:  python travel_analysis.py  (with 29_travel_bookings.csv in the same folder)
"""
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

FILE = "29_travel_bookings.csv"

# ---- Column mapping: edit the right-hand side if your headers differ ----
COLS = {
    "id": "Booking_ID",
    "dest": "Destination",
    "package": "Package_Type",
    "date": "Booking_Date",
    "days": "Trip_Duration_Days",
    "amount": "Booking_Amount",
    "travelers": "Travelers",      # may also be Num_Travelers / Number_of_Travelers
    "age": "Customer_Age",         # may also be Age
    "rating": "Customer_Rating",   # may also be Rating
}
ALT = {
    "travelers": ["Travelers", "Num_Travelers", "Number_of_Travelers", "Traveler_Count"],
    "age": ["Customer_Age", "Age"],
    "rating": ["Customer_Rating", "Rating"],
}


def section(title):
    print("\n" + "=" * 75 + f"\n{title}\n" + "=" * 75)


# ------------------------------------------------------------------
# 1. Load & inspect
# ------------------------------------------------------------------
section("1. LOAD & INSPECT")
df = pd.read_csv(FILE)
for key, options in ALT.items():           # auto-detect alternative header names
    for opt in options:
        if opt in df.columns:
            COLS[key] = opt
            break
print("Shape:", df.shape)
print("\nColumns & dtypes:\n", df.dtypes)
print("\nHead:\n", df.head())

ID, DEST, PKG, DATE = COLS["id"], COLS["dest"], COLS["package"], COLS["date"]
DAYS, AMT = COLS["days"], COLS["amount"]
TRAV = COLS["travelers"] if COLS["travelers"] in df.columns else None
AGE = COLS["age"] if COLS["age"] in df.columns else None
RATE = COLS["rating"] if COLS["rating"] in df.columns else None



1. LOAD & INSPECT
Shape: (608, 9)

Columns & dtypes:
 Booking_ID             object
Customer_Age            int64
Destination            object
Package_Type           object
Booking_Date           object
Trip_Duration_Days    float64
Travelers               int64
Booking_Amount        float64
Customer_Rating       float64
dtype: object

Head:
   Booking_ID  Customer_Age Destination Package_Type Booking_Date  Trip_Duration_Days  Travelers  Booking_Amount  Customer_Rating
0     BK1028            30      Kerala     Standard   2025-04-06                 7.0          2         70900.0              5.0
1     BK1376            48        Bali       Budget   2025-09-16                 2.0          4         51000.0              3.8
2     BK1053            43       Dubai      Premium   2025-04-08                 7.0          2        225200.0              4.0
3     BK1494            47      Kerala      Premium   2025-09-24                 5.0          2         63500.0              4.5
4     BK

In [ ]:
# ------------------------------------------------------------------
# 2. Missing values & duplicate Booking_IDs
# ------------------------------------------------------------------
section("2. MISSING VALUES & DUPLICATES")
print("Missing values per column:\n", df.isna().sum())
dup_mask = df.duplicated(subset=ID, keep=False)
print(f"\nRows sharing a duplicate Booking_ID: {dup_mask.sum()}")
print(f"Fully identical duplicate rows: {df.duplicated().sum()}")
if dup_mask.any():
    print(df[dup_mask].sort_values(ID).head(10))
before = len(df)
df = df.drop_duplicates(subset=ID, keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate Booking_ID rows (kept first occurrence).")


2. MISSING VALUES & DUPLICATES
Missing values per column:
 Booking_ID             0
Customer_Age           0
Destination            0
Package_Type           0
Booking_Date           0
Trip_Duration_Days    19
Travelers              0
Booking_Amount        22
Customer_Rating       12
dtype: int64

Rows sharing a duplicate Booking_ID: 16
Fully identical duplicate rows: 8
    Booking_ID  Customer_Age Destination Package_Type Booking_Date  Trip_Duration_Days  Travelers  Booking_Amount  Customer_Rating
288     BK1010            41      Manali      Premium   2025-11-20                 5.0          2         56800.0              4.2
494     BK1010            41      Manali      Premium   2025-11-20                 5.0          2         56800.0              4.2
258     BK1037            18      Manali    Adventure   2025-06-18                11.0          1         44800.0              3.8
386     BK1037            18      Manali    Adventure   2025-06-18                11.0          1      

In [ ]:
# ------------------------------------------------------------------
# 2. Missing values & duplicate Booking_IDs
# ------------------------------------------------------------------
section("2. MISSING VALUES & DUPLICATES")
print("Missing values per column:\n", df.isna().sum())
dup_mask = df.duplicated(subset=ID, keep=False)
print(f"\nRows sharing a duplicate Booking_ID: {dup_mask.sum()}")
print(f"Fully identical duplicate rows: {df.duplicated().sum()}")
if dup_mask.any():
    print(df[dup_mask].sort_values(ID).head(10))
before = len(df)
df = df.drop_duplicates(subset=ID, keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate Booking_ID rows (kept first occurrence).")


2. MISSING VALUES & DUPLICATES
Missing values per column:
 Booking_ID             0
Customer_Age           0
Destination            0
Package_Type           0
Booking_Date           0
Trip_Duration_Days    18
Travelers              0
Booking_Amount        22
Customer_Rating       12
dtype: int64

Rows sharing a duplicate Booking_ID: 0
Fully identical duplicate rows: 0
Removed 0 duplicate Booking_ID rows (kept first occurrence).


In [ ]:
# ------------------------------------------------------------------
# 4. Dates: month and weekday
# ------------------------------------------------------------------
section("4. DATETIME FEATURES")
df[DATE] = pd.to_datetime(df[DATE], errors="coerce")
print("Unparseable dates:", df[DATE].isna().sum())
df["Booking_Month"] = df[DATE].dt.month
df["Month_Name"] = df[DATE].dt.month_name()
df["Booking_Weekday"] = df[DATE].dt.day_name()
print(df[[DATE, "Booking_Month", "Month_Name", "Booking_Weekday"]].head())



4. DATETIME FEATURES
Unparseable dates: 0
  Booking_Date  Booking_Month Month_Name Booking_Weekday
0   2025-04-06              4      April          Sunday
1   2025-09-16              9  September         Tuesday
2   2025-04-08              4      April         Tuesday
3   2025-09-24              9  September       Wednesday
4   2025-04-25              4      April          Friday


In [ ]:
# ------------------------------------------------------------------
# 5. Totals
# ------------------------------------------------------------------
section("5. TOTALS")
total_bookings = df[ID].nunique()
total_travelers = int(df[TRAV].sum()) if TRAV else None
total_revenue = df[AMT].sum()
print(f"Total bookings   : {total_bookings:,}")
print(f"Total travelers  : {total_travelers:,}" if TRAV else "Total travelers  : column not found")
print(f"Total revenue    : {total_revenue:,.2f}")


5. TOTALS
Total bookings   : 600
Total travelers  : 1,769
Total revenue    : 93,065,800.00


In [ ]:
# ------------------------------------------------------------------
# 6. Averages
# ------------------------------------------------------------------
section("6. AVERAGES")
print(f"Average booking amount : {df[AMT].mean():,.2f}")
print(f"Average trip duration  : {df[DAYS].mean():.2f} days")



6. AVERAGES
Average booking amount : 161,013.49
Average trip duration  : 6.26 days


In [ ]:
# ------------------------------------------------------------------
# 7. Destination ranking
# ------------------------------------------------------------------
section("7. DESTINATION RANKING")
dest_rank = (df.groupby(DEST)
               .agg(Bookings=(ID, "count"), Revenue=(AMT, "sum"), Avg_Booking=(AMT, "mean"))
               .round(2))
dest_rank["Count_Rank"] = dest_rank["Bookings"].rank(ascending=False, method="min").astype(int)
dest_rank["Revenue_Rank"] = dest_rank["Revenue"].rank(ascending=False, method="min").astype(int)
dest_rank = dest_rank.sort_values(["Bookings", "Revenue"], ascending=False)
print(dest_rank)


7. DESTINATION RANKING
             Bookings     Revenue  Avg_Booking  Count_Rank  Revenue_Rank
Destination                                                             
Goa                83   7130000.0     88024.69           1             8
Kerala             81   9493000.0    118662.50           2             3
Manali             64   6234700.0     98963.49           3            10
Jaipur             58   5627700.0     98731.58           4            11
Bali               46   8022600.0    186572.09           5             6
Thailand           46   7601900.0    176788.37           5             7
Andaman            43   6387800.0    155800.00           7             9
Singapore          41  10972300.0    281341.03           8             1
Ladakh             41   4477600.0    114810.26           8            12
Dubai              38   9020000.0    243783.78          10             4
Paris              35   8519400.0    258163.64          11             5
Maldives           24   957

In [ ]:
# ------------------------------------------------------------------
# 8. Trip duration by package type
# ------------------------------------------------------------------
section("8. AVG TRIP DURATION BY PACKAGE TYPE")
pkg_summary = (df.groupby(PKG)
                 .agg(Bookings=(ID, "count"),
                      Avg_Duration=(DAYS, "mean"),
                      Avg_Amount=(AMT, "mean"),
                      Revenue=(AMT, "sum"))
                 .round(2).sort_values("Avg_Duration", ascending=False))
print(pkg_summary)



8. AVG TRIP DURATION BY PACKAGE TYPE
              Bookings  Avg_Duration  Avg_Amount     Revenue
Package_Type                                                
Luxury              50          8.83   372329.17  17871800.0
Adventure          101          7.79   178565.26  16963700.0
Premium             81          6.87   208443.59  16258600.0
Family              83          6.32   190904.88  15654200.0
Standard           162          5.86   124512.74  19548500.0
Budget             123          4.11    57364.41   6769000.0


In [ ]:
# ------------------------------------------------------------------
# 9. Age group preferences
# ------------------------------------------------------------------
section("9. AGE GROUP PREFERENCES")
top_age_dest = None
if AGE:
    df["Age_Group"] = pd.cut(df[AGE], bins=[0, 24, 34, 44, 54, 120],
                             labels=["<25", "25-34", "35-44", "45-54", "55+"])
    print("Bookings & spend by age group:")
    print(df.groupby("Age_Group", observed=True)
            .agg(Bookings=(ID, "count"), Avg_Amount=(AMT, "mean"), Avg_Days=(DAYS, "mean"))
            .round(2))
    print("\nPackage type preference (row %):")
    print((pd.crosstab(df["Age_Group"], df[PKG], normalize="index") * 100).round(1))
    print("\nTop destination per age group:")
    top_age_dest = df.groupby("Age_Group", observed=True)[DEST].agg(lambda s: s.value_counts().idxmax())
    print(top_age_dest)
else:
    print("No age column found - skipped.")


9. AGE GROUP PREFERENCES
Bookings & spend by age group:
           Bookings  Avg_Amount  Avg_Days
Age_Group                                
<25              62   171170.49      6.65
25-34           151   163552.38      6.20
35-44           203   152477.95      6.20
45-54           117   165068.18      6.29
55+              67   164484.62      6.20

Package type preference (row %):
Package_Type  Adventure  Budget  Family  Luxury  Premium  Standard
Age_Group                                                         
<25                22.6    21.0    14.5     9.7     12.9      19.4
25-34              21.2    21.9    12.6     7.9     13.2      23.2
35-44              12.3    20.2    11.8     8.4     14.3      33.0
45-54              17.1    18.8    17.9     8.5     13.7      23.9
55+                14.9    20.9    14.9     7.5     11.9      29.9

Top destination per age group:
Age_Group
<25      Kerala
25-34    Manali
35-44       Goa
45-54    Kerala
55+         Goa
Name: Destination, dtype

In [ ]:
# ------------------------------------------------------------------
# 10. Top 5 months
# ------------------------------------------------------------------
section("10. MONTHLY TREND & TOP 5 MONTHS")
monthly = (df.groupby(["Booking_Month", "Month_Name"])[ID].count()
             .rename("Bookings").reset_index())
print("Monthly trend (Jan-Dec):\n", monthly.sort_values("Booking_Month"))
top5_months = monthly.sort_values("Bookings", ascending=False).head(5)
print("\nTop 5 months by booking volume:\n", top5_months)
print("\nBookings by weekday:\n", df["Booking_Weekday"].value_counts())


10. MONTHLY TREND & TOP 5 MONTHS
Monthly trend (Jan-Dec):
     Booking_Month Month_Name  Bookings
0               1    January        32
1               2   February        18
2               3      March        42
3               4      April        67
4               5        May        89
5               6       June        51
6               7       July        42
7               8     August        42
8               9  September        45
9              10    October        46
10             11   November        51
11             12   December        75

Top 5 months by booking volume:
     Booking_Month Month_Name  Bookings
4               5        May        89
11             12   December        75
3               4      April        67
10             11   November        51
5               6       June        51

Bookings by weekday:
 Booking_Weekday
Saturday     103
Wednesday     99
Tuesday       87
Monday        85
Thursday      79
Sunday        78
Friday        69
Name: c

In [ ]:
# ------------------------------------------------------------------
# 11. List, tuple, set
# ------------------------------------------------------------------
section("11. LIST / TUPLE / SET")
popular_destinations = dest_rank.head(5).index.tolist()          # list (mutable)
package_types = tuple(sorted(df[PKG].dropna().unique()))        # tuple (fixed)
unique_destinations = set(df[DEST].dropna())                    # set (unique)
print("Popular destinations (list):", popular_destinations)
print("Package types (tuple)      :", package_types)
print(f"Unique destinations (set)  : {len(unique_destinations)} -> {sorted(unique_destinations)}")


11. LIST / TUPLE / SET
Popular destinations (list): ['Goa', 'Kerala', 'Manali', 'Jaipur', 'Bali']
Package types (tuple)      : ('Adventure', 'Budget', 'Family', 'Luxury', 'Premium', 'Standard')
Unique destinations (set)  : 12 -> ['Andaman', 'Bali', 'Dubai', 'Goa', 'Jaipur', 'Kerala', 'Ladakh', 'Maldives', 'Manali', 'Paris', 'Singapore', 'Thailand']


In [ ]:
# ------------------------------------------------------------------
# 12. Dictionary + function
# ------------------------------------------------------------------
section("12. DICTIONARY & FUNCTION")
dest_counts = df[DEST].value_counts().to_dict()
print("Destination-wise booking counts:", dest_counts)


def avg_booking_value(destination, data=df):
    """Average booking amount for one destination (case-insensitive)."""
    subset = data[data[DEST].str.lower() == str(destination).lower()]
    if subset.empty:
        return None
    return round(subset[AMT].mean(), 2)


for d in popular_destinations[:3]:
    print(f"Average booking value for {d}: {avg_booking_value(d):,.2f}")
print("Unknown destination ->", avg_booking_value("Atlantis"))



12. DICTIONARY & FUNCTION
Destination-wise booking counts: {'Goa': 83, 'Kerala': 81, 'Manali': 64, 'Jaipur': 58, 'Thailand': 46, 'Bali': 46, 'Andaman': 43, 'Singapore': 41, 'Ladakh': 41, 'Dubai': 38, 'Paris': 35, 'Maldives': 24}
Average booking value for Goa: 88,024.69
Average booking value for Kerala: 118,662.50
Average booking value for Manali: 98,963.49
Unknown destination -> None


In [ ]:
# ------------------------------------------------------------------
# 13. NumPy statistics
# ------------------------------------------------------------------
section("13. NUMPY STATISTICS (Booking_Amount)")
amounts = df[AMT].to_numpy()
print(f"Mean   : {np.mean(amounts):,.2f}")
print(f"Median : {np.median(amounts):,.2f}")
print(f"Std Dev: {np.std(amounts, ddof=1):,.2f}")


13. NUMPY STATISTICS (Booking_Amount)
Mean   : nan
Median : nan
Std Dev: nan


In [ ]:
# ------------------------------------------------------------------
# 14. IQR outliers + duration bands
# ------------------------------------------------------------------
section("14. IQR OUTLIERS & DURATION BANDS")
q1, q3 = np.percentile(amounts, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:,.2f}  Q3={q3:,.2f}  IQR={iqr:,.2f}")
print(f"Bounds: lower={lower:,.2f}  upper={upper:,.2f}")
expensive = df[df[AMT] > upper].sort_values(AMT, ascending=False)
print(f"Unusually expensive bookings: {len(expensive)} ({len(expensive)/len(df):.1%})")
print(expensive[[ID, DEST, PKG, AMT]].head(10))

# Bands: Short <= 4 days, Medium 5-9 days, Long >= 10 days (adjust to your data)
df["Duration_Band"] = pd.cut(df[DAYS], bins=[0, 4, 9, np.inf],
                             labels=["Short", "Medium", "Long"])
print("\nTrip-duration bands:")
bands = (df.groupby("Duration_Band", observed=True)
           .agg(Bookings=(ID, "count"), Avg_Amount=(AMT, "mean"), Revenue=(AMT, "sum"))
           .round(2))
print(bands)


14. IQR OUTLIERS & DURATION BANDS
Q1=nan  Q3=nan  IQR=nan
Bounds: lower=nan  upper=nan
Unusually expensive bookings: 0 (0.0%)
Empty DataFrame
Columns: [Booking_ID, Destination, Package_Type, Booking_Amount]
Index: []

Trip-duration bands:
               Bookings  Avg_Amount     Revenue
Duration_Band                                  
Short               134    58626.72   7680100.0
Medium              404   177313.14  68797500.0
Long                 44   323073.17  13246000.0


In [ ]:
# ------------------------------------------------------------------
# 15. Class + recommendations
# ------------------------------------------------------------------
section("15. TravelBooking CLASS")


class TravelBooking:
    def __init__(self, booking_id, destination, amount):
        self.booking_id = booking_id
        self.destination = destination
        self.amount = amount

    def is_premium(self, threshold=upper):
        """True if the booking exceeds the IQR upper bound."""
        return self.amount > threshold

    def __repr__(self):
        return (f"TravelBooking(id={self.booking_id}, destination={self.destination}, "
                f"amount={self.amount:,.2f})")


sample = [TravelBooking(r[ID], r[DEST], r[AMT]) for _, r in df.head(3).iterrows()]
for b in sample:
    print(b, "| premium:", b.is_premium())

section("FIVE TRAVEL-PACKAGE PLANNING RECOMMENDATIONS")
best_rev = dest_rank["Revenue"].idxmax()
best_cnt = dest_rank["Bookings"].idxmax()
peak = ", ".join(top5_months["Month_Name"].head(3))
low_month = monthly.sort_values("Bookings").iloc[0]["Month_Name"]
long_pkg = pkg_summary["Avg_Duration"].idxmax()
rich_pkg = pkg_summary["Avg_Amount"].idxmax()
recs = [
    f"Prioritise inventory and marketing for {best_cnt} (most bookings) and {best_rev} "
    f"(highest revenue); if they differ, bundle the volume leader with premium add-ons.",
    f"Plan capacity and early-bird pricing around peak months ({peak}); run "
    f"discounts or off-season bundles in {low_month} to smooth demand.",
    f"Promote the '{long_pkg}' package for longer stays and '{rich_pkg}' as the "
    f"premium tier; use Short/Medium/Long bands to design weekend, week and extended itineraries.",
    f"Create a VIP/luxury line for the {len(expensive)} high-value bookings above "
    f"{upper:,.0f} and offer personalised upsells to similar customers.",
    "Tailor packages by age segment (e.g. adventure/short trips for younger travellers, "
    "comfort/long trips for older ones) and track ratings per destination to fix weak spots."
    if AGE else
    "Collect customer age and rating data to tailor packages by segment and fix weak destinations.",
]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")

if RATE:
    section("BONUS: AVERAGE RATING BY DESTINATION")
    print(df.groupby(DEST)[RATE].agg(["mean", "count"]).round(2).sort_values("mean", ascending=False))

df.to_csv("travel_bookings_cleaned.csv", index=False)
print("\nCleaned data saved to travel_bookings_cleaned.csv")



15. TravelBooking CLASS
TravelBooking(id=BK1028, destination=Kerala, amount=70,900.00) | premium: False
TravelBooking(id=BK1376, destination=Bali, amount=51,000.00) | premium: False
TravelBooking(id=BK1053, destination=Dubai, amount=225,200.00) | premium: False

FIVE TRAVEL-PACKAGE PLANNING RECOMMENDATIONS
1. Prioritise inventory and marketing for Goa (most bookings) and Singapore (highest revenue); if they differ, bundle the volume leader with premium add-ons.
2. Plan capacity and early-bird pricing around peak months (May, December, April); run discounts or off-season bundles in February to smooth demand.
3. Promote the 'Luxury' package for longer stays and 'Luxury' as the premium tier; use Short/Medium/Long bands to design weekend, week and extended itineraries.
4. Create a VIP/luxury line for the 0 high-value bookings above nan and offer personalised upsells to similar customers.
5. Tailor packages by age segment (e.g. adventure/short trips for younger travellers, comfort/long tri